# PageZero GRPO Training (Colab) - Unsloth + Local or Deployed Backend

This notebook trains the PageZero agent with GRPO using your existing training pipeline.

You can run with any backend URL mode:
- local_same_machine: use local Docker backend on the same machine as notebook
- local_via_tunnel: use local Docker backend exposed by HTTPS tunnel (best for Colab)
- deployed_space: use deployed backend URL (HF Space or VM URL)

Only one variable controls this switch: `ENV_URL` (derived from `BACKEND_MODE`).

## 1) Install Dependencies

Includes Unsloth, TRL, vLLM, and OpenEnv runtime.

In [ ]:
%%capture
!pip install -q -U pip
!pip install -q \
    "unsloth" \
    "trl[vllm]>=0.29.0" \
    "vllm>=0.11.0" \
    "peft" \
    "transformers" \
    "datasets" \
    "accelerate" \
    "matplotlib" \
    "pandas" \
    "huggingface_hub>=0.20.0" \
    "openenv-core[core]>=0.2.1"

## 2) Configuration

Set token, backend mode, URLs, model, and training hyperparameters.

Use this backend mode selector:
- local_same_machine: local Docker backend on the same machine as notebook
- local_via_tunnel: local Docker backend exposed by HTTPS tunnel (for Colab)
- deployed_space: deployed backend URL (HF Space or VM endpoint)
- custom_url: read URL from PAGEZERO_ENV_URL environment variable

In [ ]:
import os

IN_COLAB = "COLAB_GPU" in os.environ

# --- HuggingFace token (optional unless pushing to hub) ---
try:
    from google.colab import userdata
    if "HF_TOKEN" not in os.environ:
        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
except Exception:
    pass

if "HF_TOKEN" not in os.environ:
    print("WARNING: HF_TOKEN not found. Needed only for push_to_hub.")

# --- Backend mode selector ---
BACKEND_MODE = "local_same_machine"  # local_same_machine | local_via_tunnel | deployed_space | custom_url

LOCAL_SAME_MACHINE_URL = "http://localhost:8000"
LOCAL_TUNNEL_URL = "https://YOUR-TUNNEL-URL.trycloudflare.com"
DEPLOYED_SPACE_URL = "https://YOUR_SPACE_NAME.hf.space"
CUSTOM_ENV_URL = os.environ.get("PAGEZERO_ENV_URL", "").strip()

if BACKEND_MODE == "local_same_machine":
    ENV_URL = LOCAL_SAME_MACHINE_URL
elif BACKEND_MODE == "local_via_tunnel":
    ENV_URL = LOCAL_TUNNEL_URL
elif BACKEND_MODE == "deployed_space":
    ENV_URL = DEPLOYED_SPACE_URL
elif BACKEND_MODE == "custom_url":
    if not CUSTOM_ENV_URL:
        raise ValueError("BACKEND_MODE='custom_url' but PAGEZERO_ENV_URL is empty.")
    ENV_URL = CUSTOM_ENV_URL
else:
    raise ValueError(f"Unknown BACKEND_MODE: {BACKEND_MODE}")

if IN_COLAB and BACKEND_MODE == "local_same_machine":
    print("WARNING: localhost points to the Colab VM, not your laptop.")
    print("Use local_via_tunnel or deployed_space for Colab.")

# --- Repo / checkout config ---
REPO_URL = "https://github.com/YOUR_ORG/PageZero.git"
USE_EXISTING_CHECKOUT = not IN_COLAB
EXISTING_CHECKOUT_DIR = os.getcwd() if not IN_COLAB else "/content/PageZero"
CLONE_DIR = "/content/PageZero" if IN_COLAB else EXISTING_CHECKOUT_DIR

# --- Model config ---
USE_UNSLOTH_MODEL = False  # True will try 4-bit Unsloth model id
UNSLOTH_MODEL_ID = "unsloth/Qwen3-0.6B-bnb-4bit"
BASE_MODEL_ID = "Qwen/Qwen3-0.6B"
MODEL_ID = UNSLOTH_MODEL_ID if USE_UNSLOTH_MODEL else BASE_MODEL_ID
FALLBACK_MODEL_ID = BASE_MODEL_ID

# --- Training hyperparameters ---
NUM_EPISODES = 50
NUM_GENERATIONS = 8
MAX_TURNS = 15
MAX_NEW_TOKENS = 512
VLLM_MODE = "colocate"
OUTPUT_DIR_BASE = "outputs/pagezero-colab"

# --- Hub push config ---
PUSH_TO_HUB = False
HUB_REPO = "YOUR_USERNAME/pagezero-sre-agent"

print(f"IN_COLAB      : {IN_COLAB}")
print(f"BACKEND_MODE  : {BACKEND_MODE}")
print(f"ENV_URL       : {ENV_URL}")
print(f"MODEL_ID      : {MODEL_ID}")
print(f"EPISODES      : {NUM_EPISODES}")
print(f"GENERATIONS   : {NUM_GENERATIONS}")

## 3) Project Setup

If running in Colab, clone your repository first.
If running from an existing local checkout, keep `USE_EXISTING_CHECKOUT = True`.

In [ ]:
import os
import subprocess

if USE_EXISTING_CHECKOUT:
    CLONE_DIR = EXISTING_CHECKOUT_DIR
    if not os.path.exists(CLONE_DIR):
        raise FileNotFoundError(f"Existing checkout not found: {CLONE_DIR}")
else:
    if "YOUR_ORG" in REPO_URL:
        raise ValueError("Please set REPO_URL to your real project repository URL first.")
    if not os.path.exists(CLONE_DIR):
        subprocess.run(["git", "clone", REPO_URL, CLONE_DIR], check=True)

os.chdir(CLONE_DIR)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

print(f"Working directory: {os.getcwd()}")

## 4) Smoke Test — Verify Backend Connectivity

Connect to the selected backend URL, run reset, and execute one safe action.

In [ ]:
import requests

try:
    from PageZero.client import PageZeroEnvClient
    from PageZero.models import PageZeroAction
except Exception:
    from client import PageZeroEnvClient
    from models import PageZeroAction

print(f"Testing backend: {ENV_URL}")

for suffix in ["/healthz", "/health"]:
    url = ENV_URL.rstrip("/") + suffix
    try:
        r = requests.get(url, timeout=10)
        print(f"{url} -> {r.status_code}")
    except Exception as e:
        print(f"{url} -> request failed: {e}")

env = PageZeroEnvClient(base_url=ENV_URL)
try:
    reset_result = env.reset()
    obs = reset_result.observation
    print("Reset successful")
    print(f"Step {obs.step}/{obs.max_steps}")
    print(f"Active alerts: {obs.active_alerts}")

    step_result = env.step(PageZeroAction(tool="check_alerts", args={}))
    step_obs = step_result.observation
    print(f"Step reward: {step_result.reward}")
    print(f"Done: {step_result.done}")
    print(f"SLA status: {step_obs.sla_status}")
finally:
    env.close()

## 5) Import Training Utilities (Kube-Style)

Import rollout and reward logic directly from the training module, then patch TRL/vLLM compatibility.

In [ ]:
import csv
import logging
from datetime import datetime
from pathlib import Path

from datasets import Dataset
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

try:
    from PageZero.client import PageZeroEnvClient
    from PageZero.train import (
        SYSTEM_PROMPT,
        rollout_once,
        reward_total,
        reward_diagnosis,
        reward_fix,
        plot_rewards,
        patch_trl_vllm_compat,
    )
except Exception:
    from client import PageZeroEnvClient
    from train import (
        SYSTEM_PROMPT,
        rollout_once,
        reward_total,
        reward_diagnosis,
        reward_fix,
        plot_rewards,
        patch_trl_vllm_compat,
    )

patch_trl_vllm_compat()

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

print("Imported training utilities successfully.")
print("System prompt preview:")
print(SYSTEM_PROMPT[:220] + "...")

## 6) Build GRPO Trainer

Set tokenizer, environment client, dataset, reward logging, rollout function, GRPO config, LoRA config, and trainer.

In [ ]:
# ---- Tokenizer ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ---- Environment ----
env = PageZeroEnvClient(base_url=ENV_URL)

# ---- Dataset (each entry triggers one episode) ----
dataset = Dataset.from_dict({"prompt": ["Diagnose and fix this production incident."] * NUM_EPISODES})

# ---- Output directory ----
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = Path(f"{OUTPUT_DIR_BASE}-{timestamp}")
output_dir.mkdir(parents=True, exist_ok=True)

# ---- Reward CSV logger ----
reward_log_path = output_dir / "reward_log.csv"
episode_counter = [0]
all_rewards = []

with open(reward_log_path, "w", newline="") as f:
    csv.writer(f).writerow(["episode", "total_reward", "diagnosis_reward", "fix_reward", "timestamp"])

def log_episode(total_r, diag_r, fix_r):
    episode_counter[0] += 1
    all_rewards.append(total_r)
    with open(reward_log_path, "a", newline="") as f:
        csv.writer(f).writerow([episode_counter[0], total_r, diag_r, fix_r, datetime.now().isoformat()])
    last10 = all_rewards[-10:]
    logger.info(
        f"Episode {episode_counter[0]}: reward={total_r:.2f} | "
        f"mean={sum(all_rewards)/len(all_rewards):.2f}, "
        f"mean(10)={sum(last10)/len(last10):.2f}"
    )

# ---- Rollout function (uses rollout_once from train.py) ----
def rollout_func(prompts, trainer):
    episode_prompt_ids = []
    episode_completion_ids = []
    episode_logprobs = []
    total_rewards = []
    diagnosis_rewards = []
    fix_rewards = []

    for _ in prompts:
        ep = rollout_once(
            trainer=trainer,
            env=env,
            tokenizer=tokenizer,
            system_prompt=SYSTEM_PROMPT,
            max_turns=MAX_TURNS,
        )
        episode_prompt_ids.append(ep["prompt_ids"])
        episode_completion_ids.append(ep["completion_ids"])
        episode_logprobs.append(ep["logprobs"])
        total_rewards.append(ep["total_reward"])
        diagnosis_rewards.append(ep["diagnosis_reward"])
        fix_rewards.append(ep["fix_reward"])
        log_episode(ep["total_reward"], ep["diagnosis_reward"], ep["fix_reward"])

    return {
        "prompt_ids": episode_prompt_ids,
        "completion_ids": episode_completion_ids,
        "logprobs": episode_logprobs,
        "total_reward": total_rewards,
        "diagnosis_reward": diagnosis_rewards,
        "fix_reward": fix_rewards,
    }

# ---- GRPO Config ----
grpo_config = GRPOConfig(
    use_vllm=True,
    vllm_mode=VLLM_MODE,
    output_dir=str(output_dir),
    num_train_epochs=1,
    learning_rate=2e-6,
    lr_scheduler_type="cosine",
    warmup_steps=2,
    max_grad_norm=1.0,
    gradient_accumulation_steps=8,
    per_device_train_batch_size=1,
    generation_batch_size=NUM_GENERATIONS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_NEW_TOKENS,
    logging_steps=1,
    save_strategy="steps",
    save_steps=10,
    temperature=1.0,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    save_total_limit=3,
    loss_type="dapo",
    mask_truncated_completions=True,
    beta=0.01,
)

# ---- LoRA Config ----
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# ---- Create trainer ----
trainer = GRPOTrainer(
    model=MODEL_ID,
    processing_class=tokenizer,
    reward_funcs=[reward_total, reward_diagnosis, reward_fix],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
    peft_config=peft_config,
)

print(f"Trainer initialized. Output path: {output_dir}")

## 7) Train, Plot Rewards, and Optional Hub Push

Run training, save model, generate reward plot, show evidence table, and optionally push adapter/checkpoints to Hub.

In [ ]:
import pandas as pd
from IPython.display import display, Image

print("Starting GRPO training...")
print(f"  Model       : {MODEL_ID}")
print(f"  Episodes    : {NUM_EPISODES}")
print(f"  Generations : {NUM_GENERATIONS}")
print(f"  Environment : {ENV_URL}")
print()

try:
    trainer.train()
finally:
    env.close()
    try:
        plot_rewards(reward_log_path, output_dir / "reward_plot.png")
    except Exception as e:
        print(f"WARNING: could not generate reward plot: {e}")

trainer.save_model(str(output_dir))
print(f"Model saved to {output_dir}")

# Show evidence table
if reward_log_path.exists():
    df = pd.read_csv(reward_log_path)
    display(df.tail(10))

# Show reward image if produced
reward_png = output_dir / "reward_plot.png"
if reward_png.exists():
    display(Image(filename=str(reward_png)))

# Optional push to Hub
if PUSH_TO_HUB:
    if "HF_TOKEN" not in os.environ:
        raise RuntimeError("HF_TOKEN not set. Add it in Colab secrets or env vars.")
    trainer.push_to_hub(repo_id=HUB_REPO)
    print(f"Model pushed to https://huggingface.co/{HUB_REPO}")
else:
    print("Skipping push. Set PUSH_TO_HUB=True to enable.")